# Data profile

A look at the raw data before it goes into the graph: what each table holds, which values appear, what is missing, and whether the tables point at each other correctly.

This notebook reads the CSV files directly with pandas. It does not use the project's loader or Neo4j, so it is an independent check of what the loader assumes.

Column descriptions are in `data/DATA_README.md`. The data snapshot was taken **2026-10-26 12:00 UTC**, which counts as "now".

In [1]:
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)

DATA = Path("../data") if Path("../data").exists() else Path("data")

FILES = ["vessels", "ports", "voyages", "port_calls", "incidents", "congestion", "bookings"]

# Only a truly empty cell counts as missing. Text such as "NA" stays text.
tables = {
    name: pd.read_csv(DATA / f"{name}.csv", dtype=str, keep_default_na=False, na_values=[""])
    for name in FILES
}

## 1. Size of each table

In [2]:
overview = pd.DataFrame(
    {
        "rows": {name: len(df) for name, df in tables.items()},
        "columns": {name: len(df.columns) for name, df in tables.items()},
    }
)
overview.loc["TOTAL"] = overview.sum()
overview

,rows,columns
vessels,40,6
ports,15,4
voyages,121,11
port_calls,231,9
incidents,90,8
congestion,578,6
bookings,400,6
TOTAL,1475,50


The total should be **1,475 rows** The graph must end up with one node per row.

## 2. Missing values

Only columns with at least one blank cell are listed. The graph keeps a blank as empty (null), never as 0.

In [3]:
missing = [
    {"table": name, "column": col, "blank": int(df[col].isna().sum()), "of rows": len(df)}
    for name, df in tables.items()
    for col in df.columns
    if df[col].isna().any()
]
pd.DataFrame(missing)

,table,column,blank,of rows
0,port_calls,anchorage_arrival_utc,118,231
1,port_calls,berth_scheduled_utc,118,231
2,port_calls,berthing_utc,119,231
3,port_calls,departure_utc,1,231
4,incidents,attributed_delay_hours,38,90


Blanks appear in only two places:

- `incidents.attributed_delay_hours`: the delay caused by the incident is not known. It is not zero.
- `port_calls` times: the event has not happened yet (a ship still at anchor has no berthing time), or the time does not apply to that kind of call (a departure call has no anchorage arrival).

## 3. Unique values per column

For each table: how many distinct values every column has. Columns with few distinct values (at most 20) are categories, and their values are listed with counts.

In [4]:
def profile(name, max_listed=20):
    df = tables[name]
    summary = pd.DataFrame(
        {
            "distinct": df.nunique(),
            "blank": df.isna().sum(),
            "example": df.iloc[0],
        }
    )
    display(summary)
    for col in df.columns:
        if df[col].nunique() <= max_listed:
            counts = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="rows")
            display(counts)

### vessels.csv

40 vessels. `vessel_class` and `technical_manager` repeat across vessels. They stay as vessel properties rather than separate nodes, because questions only compare them for equality.

In [5]:
profile("vessels")

,distinct,blank,example
vessel_id,40,0,V001
name,40,0,MV Ocean Star
imo,40,0,9876543
type,3,0,Container ship
vessel_class,5,0,Meridian 8500
technical_manager,5,0,Harbourline Ship Management


,type,rows
0,Container ship,32
1,Feeder,6
2,Bulk carrier,2


,vessel_class,rows
0,Atlas 6000,15
1,Meridian 8500,9
2,Kestrel 4000,8
3,Coastal 1700,6
4,Titan Bulk 82,2


,technical_manager,rows
0,Eastpoint Technical Services,14
1,Nordwave Marine,8
2,Sulamar Gestão Naval,7
3,Bluefin Fleet Services,6
4,Harbourline Ship Management,5


### ports.csv

15 ports, each with an IANA time zone. These zones convert every local time to UTC.

In [6]:
profile("ports")

,distinct,blank,example
port_code,15,0,SGSIN
name,15,0,Port of Singapore
country,15,0,Singapore
timezone,15,0,Asia/Singapore


,port_code,rows
0,SGSIN,1
1,NLRTM,1
2,DEHAM,1
3,BEANR,1
4,GBFXT,1
5,CNSHA,1
6,KRPUS,1
7,AEJEA,1
8,LKCMB,1
9,BRSSZ,1


,name,rows
0,Port of Singapore,1
1,Port of Rotterdam,1
2,Port of Hamburg,1
3,Port of Antwerp-Bruges,1
4,Port of Felixstowe,1
5,Port of Shanghai,1
6,Port of Busan,1
7,Jebel Ali,1
8,Port of Colombo,1
9,Port of Santos,1


,country,rows
0,Singapore,1
1,Netherlands,1
2,Germany,1
3,Belgium,1
4,United Kingdom,1
5,China,1
6,South Korea,1
7,United Arab Emirates,1
8,Sri Lanka,1
9,Brazil,1


,timezone,rows
0,Asia/Singapore,1
1,Europe/Amsterdam,1
2,Europe/Berlin,1
3,Europe/Brussels,1
4,Europe/London,1
5,Asia/Shanghai,1
6,Asia/Seoul,1
7,Asia/Dubai,1
8,Asia/Colombo,1
9,America/Sao_Paulo,1


### voyages.csv

`status` shows how many voyages are still under way. Times in this table are **local** to a port.

In [7]:
profile("voyages")

,distinct,blank,example
voyage_id,121,0,VG-0981
vessel_id,40,0,V001
origin,15,0,NLRTM
destination,15,0,SGSIN
departure_local,118,0,2026-07-18 12:00
original_eta_local,119,0,2026-08-14 10:00
revised_eta_local,119,0,2026-08-14 19:00
berth_window_start_local,119,0,2026-08-14 08:00
berth_window_end_local,119,0,2026-08-14 20:00
status,4,0,Completed


,origin,rows
0,AEJEA,12
1,SGSIN,11
2,CNSHA,11
3,BEANR,10
4,GBFXT,10
5,LKCMB,9
6,MYPKG,9
7,BRSSZ,8
8,NLRTM,7
9,USNYC,7


,destination,rows
0,AEJEA,15
1,NLRTM,12
2,GBFXT,10
3,BRSSZ,10
4,SGSIN,8
5,USNYC,8
6,MYPKG,8
7,CNSHA,8
8,DEHAM,7
9,BEANR,7


,status,rows
0,Completed,112
1,On schedule,4
2,Scheduled,3
3,Delayed,2


,recorded_delay_hours,rows
0,0,40
1,9,17
2,2,15
3,4,11
4,6,11
5,31,10
6,14,6
7,20,5
8,55,1
9,3,1


### port_calls.csv

Times in this table are already **UTC**.

In [8]:
profile("port_calls")

,distinct,blank,example
call_id,231,0,PC-7001
voyage_id,118,0,VG-0981
vessel_id,40,0,V001
port,15,0,NLRTM
call_type,2,0,Departure
anchorage_arrival_utc,110,118,NaN
berth_scheduled_utc,112,118,NaN
berthing_utc,111,119,NaN
departure_utc,222,1,2026-07-18T10:00Z


,port,rows
0,AEJEA,27
1,GBFXT,20
2,CNSHA,19
3,SGSIN,18
4,BEANR,17
5,BRSSZ,17
6,MYPKG,17
7,USNYC,15
8,LKCMB,15
9,INNSA,14


,call_type,rows
0,Departure,118
1,Arrival,113


### incidents.csv

`status` and `source` matter most: they decide whether an incident counts as verified, and who reported it.

In [9]:
profile("incidents")

,distinct,blank,example
incident_id,90,0,INC-201
voyage_id,87,0,VG-1042
type,15,0,Engine cooling fault
status,4,0,Confirmed
severity,4,0,Medium
reported_utc,89,0,2026-10-06T06:00Z
source,5,0,Master's report
attributed_delay_hours,12,38,12


,type,rows
0,Mooring line failure,9
1,Adverse weather / rerouting,8
2,Hull fouling / speed loss,8
3,Auxiliary engine fault,8
4,Pilot boarding delay,8
5,Documentation query,8
6,Bunkering delay,7
7,Fog / restricted visibility,7
8,Engine cooling fault,6
9,Crew change delay,5


,status,rows
0,Confirmed,52
1,Reported,19
2,Under review,13
3,Dismissed,6


,severity,rows
0,Low,42
1,Medium,29
2,High,18
3,Unknown,1


,source,rows
0,Crew message,23
1,Agent email,18
2,Technical manager,18
3,Master's report,17
4,Port agent,14


,attributed_delay_hours,rows
0,NaN,38
1,0,15
2,3,8
3,4,8
4,6,7
5,2,6
6,8,2
7,12,1
8,9,1
9,10,1


### congestion.csv

One row per port per day. `avg_wait_hours` is a port-wide average, never a figure for a single vessel.

In [10]:
profile("congestion")

,distinct,blank,example
port,15,0,SGSIN
date,40,0,2026-09-17
congestion_level,3,0,Low
avg_wait_hours,30,0,4
vessels_at_anchor,24,0,2
source,1,0,Port authority feed


,port,rows
0,SGSIN,40
1,NLRTM,40
2,DEHAM,40
3,BEANR,40
4,GBFXT,40
5,CNSHA,40
6,KRPUS,40
7,AEJEA,40
8,LKCMB,40
9,USNYC,40


,congestion_level,rows
0,Moderate,308
1,Low,243
2,High,27


,source,rows
0,Port authority feed,578


### bookings.csv

`status` is free text. Only one row names the voyage a booking was rolled to.

In [11]:
profile("bookings")

,distinct,blank,example
booking_id,400,0,BK-50001
customer,30,0,Halden Retail
voyage_id,114,0,VG-1042
containers,57,0,40
status,4,0,Confirmed
promised_delivery_date,136,0,2026-10-28


,status,rows
0,Confirmed,331
1,Rolled to next sailing,38
2,Cancelled,30
3,Rolled to VG-1107,1


## 4. IDs are unique

Each table's ID becomes a node's `evidence_id`, which answers cite. A duplicate would make a citation ambiguous.

In [12]:
keys = {
    "vessels": ["vessel_id"],
    "ports": ["port_code"],
    "voyages": ["voyage_id"],
    "port_calls": ["call_id"],
    "incidents": ["incident_id"],
    "congestion": ["port", "date"],
    "bookings": ["booking_id"],
}
pd.DataFrame(
    [
        {"table": name, "key": " + ".join(cols), "duplicates": int(tables[name].duplicated(cols).sum())}
        for name, cols in keys.items()
    ]
)

,table,key,duplicates
0,vessels,vessel_id,0
1,ports,port_code,0
2,voyages,voyage_id,0
3,port_calls,call_id,0
4,incidents,incident_id,0
5,congestion,port + date,0
6,bookings,booking_id,0


## 5. References between tables

Each of these columns points at a row in another table, and each becomes a relationship in the graph (for example, an incident `ON_VOYAGE` a voyage). A reference to a row that does not exist would leave a broken link.

In [13]:
v, p, vo = tables["vessels"], tables["ports"], tables["voyages"]
pc, inc, cong, b = tables["port_calls"], tables["incidents"], tables["congestion"], tables["bookings"]
references = [
    ("voyages.vessel_id", vo["vessel_id"], v["vessel_id"], "Voyage -OPERATED_BY-> Vessel"),
    ("voyages.origin", vo["origin"], p["port_code"], "Voyage -FROM-> Port"),
    ("voyages.destination", vo["destination"], p["port_code"], "Voyage -TO-> Port"),
    ("port_calls.voyage_id", pc["voyage_id"], vo["voyage_id"], "PortCall -ON_VOYAGE-> Voyage"),
    ("port_calls.port", pc["port"], p["port_code"], "PortCall -AT_PORT-> Port"),
    ("port_calls.vessel_id", pc["vessel_id"], v["vessel_id"], "(kept as a property)"),
    ("incidents.voyage_id", inc["voyage_id"], vo["voyage_id"], "Incident -ON_VOYAGE-> Voyage"),
    ("congestion.port", cong["port"], p["port_code"], "CongestionSnapshot -AT_PORT-> Port"),
    ("bookings.voyage_id", b["voyage_id"], vo["voyage_id"], "Booking -ON_VOYAGE-> Voyage"),
]
pd.DataFrame(
    [
        {"column": col, "becomes": rel, "broken references": int((~values.isin(targets)).sum())}
        for col, values, targets, rel in references
    ]
)

,column,becomes,broken references
0,voyages.vessel_id,Voyage -OPERATED_BY-> Vessel,0
1,voyages.origin,Voyage -FROM-> Port,0
2,voyages.destination,Voyage -TO-> Port,0
3,port_calls.voyage_id,PortCall -ON_VOYAGE-> Voyage,0
4,port_calls.port,PortCall -AT_PORT-> Port,0
5,port_calls.vessel_id,(kept as a property),0
6,incidents.voyage_id,Incident -ON_VOYAGE-> Voyage,0
7,congestion.port,CongestionSnapshot -AT_PORT-> Port,0
8,bookings.voyage_id,Booking -ON_VOYAGE-> Voyage,0


Every count should be 0: no table points at a row that does not exist.

## 6. Booking statuses

The graph keeps the original text and adds a normalized status. A `ROLLED_TO` link is created only when the text names a voyage.

In [14]:
statuses = b["status"].value_counts().rename_axis("status").reset_index(name="bookings")
statuses["normalized"] = statuses["status"].str.split().str[0].str.lower()
statuses["names a voyage"] = statuses["status"].str.contains(r"VG-\d+", regex=True)
statuses

,status,bookings,normalized,names a voyage
0,Confirmed,331,confirmed,False
1,Rolled to next sailing,38,rolled,False
2,Cancelled,30,cancelled,False
3,Rolled to VG-1107,1,rolled,True


In [15]:
b[b["status"].str.contains("VG-")]

,booking_id,customer,voyage_id,containers,status,promised_delivery_date
7,BK-50008,Northgate Auto,VG-1042,50,Rolled to VG-1107,2026-10-28


## 7. Incident status and source

What the statuses mean:

| Status | Meaning in the assistant |
|---|---|
| Confirmed | It happened. Counted as confirmed. |
| Reported | Someone said it happened; not verified. |
| Under review | Being checked; not verified. |
| Dismissed | Checked and found not to have happened. Not counted as an occurrence. |

`source` says who reported it. The master and the crew of one ship count as **one** source, so they cannot confirm each other.

In [16]:
pd.crosstab(inc["type"], inc["status"], margins=True)

status,Confirmed,Dismissed,Reported,Under review,All
type,,,,,
Adverse weather / rerouting,3,1,3,1,8
Auxiliary engine fault,4,1,3,0,8
Bunkering delay,3,0,2,2,7
Crane breakdown,2,1,0,1,4
Crew change delay,4,0,1,0,5
Customs documentation query,0,0,0,1,1
Documentation query,2,2,3,1,8
Engine cooling fault,5,1,0,0,6
Fog / restricted visibility,5,0,1,1,7


In [17]:
pd.crosstab(inc["source"], inc["status"], margins=True)

status,Confirmed,Dismissed,Reported,Under review,All
source,,,,,
Agent email,10,0,5,3,18
Crew message,12,3,3,5,23
Master's report,12,1,4,0,17
Port agent,8,2,3,1,14
Technical manager,10,0,4,4,18
All,52,6,19,13,90


## 8. Time zones and daylight saving

Voyage times are local. Departure uses the origin port's zone; ETAs and the berth window use the destination port's zone.

VG-1042's original ETA is before the clocks went back in Rotterdam (25 Oct, 03:00) and its revised ETA is after. Subtracting the local clock times gives 54 hours; the real elapsed time is 55 hours.

In [18]:
zone = dict(zip(p["port_code"], p["timezone"]))


def to_utc(text, port):
    if pd.isna(text):
        return pd.NaT
    return pd.Timestamp(text).tz_localize(ZoneInfo(zone[port])).tz_convert("UTC")


vg = vo.set_index("voyage_id").loc["VG-1042"]
original = to_utc(vg["original_eta_local"], vg["destination"])
revised = to_utc(vg["revised_eta_local"], vg["destination"])
wall_clock = (pd.Timestamp(vg["revised_eta_local"]) - pd.Timestamp(vg["original_eta_local"])) / pd.Timedelta(hours=1)

pd.Series(
    {
        "original ETA (local, Rotterdam)": vg["original_eta_local"],
        "original ETA (UTC)": original,
        "revised ETA (local, Rotterdam)": vg["revised_eta_local"],
        "revised ETA (UTC)": revised,
        "local clock difference (h)": wall_clock,
        "real elapsed (h)": (revised - original) / pd.Timedelta(hours=1),
        "recorded_delay_hours": vg["recorded_delay_hours"],
    }
)

original ETA (local, Rotterdam)             2026-10-24 20:00
original ETA (UTC)                 2026-10-24 18:00:00+00:00
revised ETA (local, Rotterdam)              2026-10-27 02:00
revised ETA (UTC)                  2026-10-27 01:00:00+00:00
local clock difference (h)                              54.0
real elapsed (h)                                        55.0
recorded_delay_hours                                      55
dtype: object

The same check for every voyage: `recorded_delay_hours` should equal the real elapsed hours between the UTC ETAs.

In [19]:
elapsed = [
    (to_utc(r.revised_eta_local, r.destination) - to_utc(r.original_eta_local, r.destination)) / pd.Timedelta(hours=1)
    for r in vo.itertuples()
]
checked = vo.assign(elapsed=elapsed)
mismatch = checked[checked["elapsed"] != checked["recorded_delay_hours"].astype(float)]
print(f"{len(vo)} voyages checked, {len(mismatch)} mismatches")
mismatch[["voyage_id", "original_eta_local", "revised_eta_local", "recorded_delay_hours", "elapsed"]]

121 voyages checked, 0 mismatches


,voyage_id,original_eta_local,revised_eta_local,recorded_delay_hours,elapsed


## 9. Look-alike vessel names

Messages refer to vessels by name, sometimes loosely. These names are close but belong to different ships, and must stay separate.

In [20]:
v[v["name"].str.contains("Ocean")][["vessel_id", "name", "imo", "vessel_class", "technical_manager"]]

,vessel_id,name,imo,vessel_class,technical_manager
0,V001,MV Ocean Star,9876543,Meridian 8500,Harbourline Ship Management
2,V003,MV Ocean Starlight,9654321,Atlas 6000,Nordwave Marine
3,V004,MV Ocean Pearl,9876551,Meridian 8500,Harbourline Ship Management


## 10. Messages

Plain-text emails, noon reports and crew messages. The 15 in `core` are always loaded; the 45 in `extra` are optional. Every message has the same four headers: From, To, Date and Subject.

In [21]:
messages = sorted((DATA / "messages").glob("*/*.txt"))
pd.Series([m.parent.name for m in messages]).value_counts().rename_axis("collection").reset_index(name="files")

,collection,files
0,extra,45
1,core,15


In [22]:
example = DATA / "messages/core/noon_2026-10-06_ocean_star.txt"
print(example.read_text(encoding="utf-8"))

From: Master, M/V OCEAN STAR
To: operations@ops-desk.example
Date: 2026-10-06 06:00 UTC
Subject: NOON REPORT - M/V OCEAN STAR - VOY 1042

Position 07-12N 078-40E. Course 285. Avg speed last 24h 9.8 kn (planned 17.5).
Main engine stopped 0130-1330 LT due to seawater cooling pump mechanical seal failure. Spare seal fitted, ME restarted and tested OK.
Time lost: 12 hours. Engineers monitoring cooling water temperatures.
Master

